In [37]:
import pandas as pd
import numpy as np
import re
import os
from scipy.interpolate import interp1d
from scipy.interpolate import PchipInterpolator

df = pd.read_csv("../data/raw/dataset.csv")
original = df.copy()

ce_cols = [c for c in df.columns if c.endswith("CE")]
pe_cols = [c for c in df.columns if c.endswith("PE")]

def get_strike(col):
    return int(re.search(r'JAN26(\d+)(CE|PE)$', col).group(1))

ce_cols = sorted(ce_cols, key=get_strike)
pe_cols = sorted(pe_cols, key=get_strike)

ce_strikes = np.array([get_strike(c) for c in ce_cols])
pe_strikes = np.array([get_strike(c) for c in pe_cols])

print(ce_strikes)
print(pe_strikes)

[25200 25300 25400 25500 25600 25700 25800 25900 26000 26100 26200 26300
 26400 26500]
[23800 23900 24000 24100 24200 24300 24400 24500 24600 24700 24800 24900
 25000 25100]


In [24]:
filled = df.copy()

def interpolate_row(row, cols, strikes):
    values = row[cols].astype(float).values.copy() #values became a read-only NumPy array from the pandas row so added .copy()
    mask = ~np.isnan(values)

    if mask.sum() == 0:
        return values

    if mask.sum() == 1:
        values[~mask] = values[mask][0]
        return values

    kind ="linear"

    f = interp1d(
        strikes[mask],
        values[mask],
        kind=kind,
        fill_value="extrapolate",
        bounds_error=False
    )

    values[~mask] = f(strikes[~mask])
    return values

In [25]:
for idx in filled.index:
    filled.loc[idx, ce_cols] = interpolate_row(filled.loc[idx], ce_cols, ce_strikes)
    filled.loc[idx, pe_cols] = interpolate_row(filled.loc[idx], pe_cols, pe_strikes)

# time-based fallback
for col in ce_cols + pe_cols:
    filled[col] = filled[col].interpolate(method="linear", limit_direction="both")

# IV cannot be negative
filled[ce_cols + pe_cols] = filled[ce_cols + pe_cols].clip(lower=0.001)

print("Remaining missing:", filled[ce_cols + pe_cols].isna().sum().sum())

Remaining missing: 0


In [34]:
rows = []

for idx in original.index:
    dt = original.loc[idx, "datetime"]

    for col in ce_cols + pe_cols:
        if pd.isna(original.loc[idx, col]):
            rows.append({
                "id": f"{dt}||{col}",
                "value": filled.loc[idx, col]
            })

submission = pd.DataFrame(rows)

os.makedirs("../outputs/submissions", exist_ok=True)
submission.to_csv("../outputs/submissions/submission_linear_interpolation.csv", index=False)

print(submission.shape)
print(submission.head())

(5460, 2)
                                      id    value
0  07-01-2026 09:15||NIFTY27JAN2625500CE  0.11373
1  07-01-2026 09:15||NIFTY27JAN2625800CE  0.10150
2  07-01-2026 09:15||NIFTY27JAN2624100PE  0.16344
3  07-01-2026 09:20||NIFTY27JAN2625300CE  0.09681
4  07-01-2026 09:20||NIFTY27JAN2625400CE  0.10730


## Time-first interpolation

In [27]:
time_filled = original.copy()

for col in ce_cols + pe_cols:
    time_filled[col] = time_filled[col].interpolate(
        method="linear",
        limit_direction="both"
    )

print("Remaining missing:", time_filled[ce_cols + pe_cols].isna().sum().sum())

Remaining missing: 0


In [28]:
rows = []

for idx in original.index:
    dt = original.loc[idx, "datetime"]
    for col in ce_cols + pe_cols:
        if pd.isna(original.loc[idx, col]):
            rows.append({
                "id": f"{dt}||{col}",
                "value": time_filled.loc[idx, col]
            })

submission_time = pd.DataFrame(rows)
submission_time.to_csv("../outputs/submissions/submission_time_first.csv", index=False)

print(submission_time.shape)
submission_time.head()

(5460, 2)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2625500CE,0.117790
1,07-01-2026 09:15||NIFTY27JAN2625800CE,0.092160
2,07-01-2026 09:15||NIFTY27JAN2624100PE,0.165340
3,07-01-2026 09:20||NIFTY27JAN2625300CE,0.118340
4,07-01-2026 09:20||NIFTY27JAN2625400CE,0.106275


## Strike + time blend

In [29]:
def create_blend_submission(strike_weight, filename):
    blended = original.copy()
    
    for col in ce_cols + pe_cols:
        blended[col] = (
            strike_weight * filled[col] +
            (1 - strike_weight) * time_filled[col]
        )
    
    rows = []
    for idx in original.index:
        dt = original.loc[idx, "datetime"]
        for col in ce_cols + pe_cols:
            if pd.isna(original.loc[idx, col]):
                rows.append({
                    "id": f"{dt}||{col}",
                    "value": blended.loc[idx, col]
                })
    
    sub = pd.DataFrame(rows)
    sub.to_csv(f"../outputs/submissions/{filename}", index=False)
    print(filename, sub.shape)
    return sub

In [30]:
sub_blend_80 = create_blend_submission(0.8, "submission_blend_80_strike_20_time.csv")
sub_blend_70 = create_blend_submission(0.7, "submission_blend_70_strike_30_time.csv")
sub_blend_50 = create_blend_submission(0.5, "submission_blend_50_strike_50_time.csv")

submission_blend_80_strike_20_time.csv (5460, 2)
submission_blend_70_strike_30_time.csv (5460, 2)
submission_blend_50_strike_50_time.csv (5460, 2)


## Local neighbor average

In [31]:
local_filled = original.copy()

def local_neighbor_fill(row, cols):
    values = row[cols].astype(float).values.copy()
    
    for i in range(len(values)):
        if np.isnan(values[i]):
            neighbors = []
            
            left = i - 1
            while left >= 0:
                if not np.isnan(values[left]):
                    neighbors.append(values[left])
                    break
                left -= 1
            
            right = i + 1
            while right < len(values):
                if not np.isnan(values[right]):
                    neighbors.append(values[right])
                    break
                right += 1
            
            if len(neighbors) > 0:
                values[i] = np.mean(neighbors)
    
    return values

In [32]:
for idx in local_filled.index:
    local_filled.loc[idx, ce_cols] = local_neighbor_fill(local_filled.loc[idx], ce_cols)
    local_filled.loc[idx, pe_cols] = local_neighbor_fill(local_filled.loc[idx], pe_cols)

for col in ce_cols + pe_cols:
    local_filled[col] = local_filled[col].interpolate(method="linear", limit_direction="both")

print("Remaining missing:", local_filled[ce_cols + pe_cols].isna().sum().sum())

Remaining missing: 0


In [33]:
rows = []

for idx in original.index:
    dt = original.loc[idx, "datetime"]
    for col in ce_cols + pe_cols:
        if pd.isna(original.loc[idx, col]):
            rows.append({
                "id": f"{dt}||{col}",
                "value": local_filled.loc[idx, col]
            })

submission_local = pd.DataFrame(rows)
submission_local.to_csv("../outputs/submissions/submission_local_neighbor.csv", index=False)

print(submission_local.shape)
submission_local.head()

(5460, 2)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2625500CE,0.113730
1,07-01-2026 09:15||NIFTY27JAN2625800CE,0.101500
2,07-01-2026 09:15||NIFTY27JAN2624100PE,0.163440
3,07-01-2026 09:20||NIFTY27JAN2625300CE,0.102055
4,07-01-2026 09:20||NIFTY27JAN2625400CE,0.109923


## Experiment 11 - PCHIP Interpolation

In [35]:
def pchip_row(row, cols, strikes):
    values = row[cols].astype(float).values.copy()

    mask = ~np.isnan(values)

    if mask.sum() == 0:
        return values

    if mask.sum() == 1:
        values[~mask] = values[mask][0]
        return values

    f = PchipInterpolator(
        strikes[mask],
        values[mask],
        extrapolate=True
    )

    values[~mask] = f(strikes[~mask])

    return values

In [38]:
pchip_filled = original.copy()

for idx in pchip_filled.index:

    pchip_filled.loc[idx, ce_cols] = pchip_row(
        pchip_filled.loc[idx],
        ce_cols,
        ce_strikes
    )

    pchip_filled.loc[idx, pe_cols] = pchip_row(
        pchip_filled.loc[idx],
        pe_cols,
        pe_strikes
    )

for col in ce_cols + pe_cols:
    pchip_filled[col] = pchip_filled[col].interpolate(
        method="linear",
        limit_direction="both"
    )

print(
    "Remaining missing:",
    pchip_filled[ce_cols + pe_cols].isna().sum().sum()
)

Remaining missing: 0


In [39]:
rows = []

for idx in original.index:

    dt = original.loc[idx, "datetime"]

    for col in ce_cols + pe_cols:

        if pd.isna(original.loc[idx, col]):

            rows.append({
                "id": f"{dt}||{col}",
                "value": pchip_filled.loc[idx, col]
            })

submission_pchip = pd.DataFrame(rows)

submission_pchip.to_csv(
    "../outputs/submissions/submission_pchip.csv",
    index=False
)

print(submission_pchip.shape)
submission_pchip.head()

(5460, 2)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2625500CE,0.113567
1,07-01-2026 09:15||NIFTY27JAN2625800CE,0.100968
2,07-01-2026 09:15||NIFTY27JAN2624100PE,0.163830
3,07-01-2026 09:20||NIFTY27JAN2625300CE,0.104578
4,07-01-2026 09:20||NIFTY27JAN2625400CE,0.114681


## Experiment 12 - CE PCHIP PE Linear

In [40]:
mixed1 = original.copy()

for idx in mixed1.index:

    mixed1.loc[idx, ce_cols] = pchip_row(
        mixed1.loc[idx],
        ce_cols,
        ce_strikes
    )

    mixed1.loc[idx, pe_cols] = interpolate_row(
        mixed1.loc[idx],
        pe_cols,
        pe_strikes
    )

In [47]:
rows = []

for idx in original.index:

    dt = original.loc[idx, "datetime"]

    for col in ce_cols + pe_cols:

        if pd.isna(original.loc[idx, col]):

            rows.append({
                "id": f"{dt}||{col}",
                "value": mixed1.loc[idx, col]
            })

submission_mixed1 = pd.DataFrame(rows)

submission_mixed1.to_csv(
    "../outputs/submissions/submission_ce_pchip_pe_linear.csv",
    index=False
)

print(submission_mixed1.shape)
submission_mixed1.head()

(5460, 2)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2625500CE,0.113567
1,07-01-2026 09:15||NIFTY27JAN2625800CE,0.100968
2,07-01-2026 09:15||NIFTY27JAN2624100PE,0.163440
3,07-01-2026 09:20||NIFTY27JAN2625300CE,0.104578
4,07-01-2026 09:20||NIFTY27JAN2625400CE,0.114681


## Experiment 13 - CE Linear PE PCHIP

In [48]:
mixed2 = original.copy()

for idx in mixed2.index:

    mixed2.loc[idx, ce_cols] = interpolate_row(
        mixed2.loc[idx],
        ce_cols,
        ce_strikes
    )

    mixed2.loc[idx, pe_cols] = pchip_row(
        mixed2.loc[idx],
        pe_cols,
        pe_strikes
    )

In [50]:
rows = []

for idx in original.index:

    dt = original.loc[idx, "datetime"]

    for col in ce_cols + pe_cols:

        if pd.isna(original.loc[idx, col]):

            rows.append({
                "id": f"{dt}||{col}",
                "value": mixed2.loc[idx, col]
            })

submission_mixed2 = pd.DataFrame(rows)

submission_mixed2.to_csv(
    "../outputs/submissions/submission_ce_linear_pe_pchip.csv",
    index=False
)

print(submission_mixed2.shape)
submission_mixed2.head()

(5460, 2)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2625500CE,0.11373
1,07-01-2026 09:15||NIFTY27JAN2625800CE,0.10150
2,07-01-2026 09:15||NIFTY27JAN2624100PE,0.16383
3,07-01-2026 09:20||NIFTY27JAN2625300CE,0.09681
4,07-01-2026 09:20||NIFTY27JAN2625400CE,0.10730


## Experiment 15 - Moneyness Linear Interpolation

In [51]:
def moneyness_row(row, cols, strikes, spot):
    values = row[cols].astype(float).values.copy()

    mask = ~np.isnan(values)

    if mask.sum() == 0:
        return values

    if mask.sum() == 1:
        values[~mask] = values[mask][0]
        return values

    moneyness = strikes / spot

    f = interp1d(
        moneyness[mask],
        values[mask],
        kind="linear",
        fill_value="extrapolate",
        bounds_error=False
    )

    values[~mask] = f(moneyness[~mask])

    return values

In [52]:
moneyness_filled = original.copy()

for idx in moneyness_filled.index:

    spot = moneyness_filled.loc[idx, "underlying_price"]

    moneyness_filled.loc[idx, ce_cols] = moneyness_row(
        moneyness_filled.loc[idx],
        ce_cols,
        ce_strikes,
        spot
    )

    moneyness_filled.loc[idx, pe_cols] = moneyness_row(
        moneyness_filled.loc[idx],
        pe_cols,
        pe_strikes,
        spot
    )

for col in ce_cols + pe_cols:
    moneyness_filled[col] = moneyness_filled[col].interpolate(
        method="linear",
        limit_direction="both"
    )

print(
    "Remaining missing:",
    moneyness_filled[ce_cols + pe_cols].isna().sum().sum()
)

Remaining missing: 0


In [53]:
rows = []

for idx in original.index:

    dt = original.loc[idx, "datetime"]

    for col in ce_cols + pe_cols:

        if pd.isna(original.loc[idx, col]):

            rows.append({
                "id": f"{dt}||{col}",
                "value": moneyness_filled.loc[idx, col]
            })

submission_moneyness = pd.DataFrame(rows)

submission_moneyness.to_csv(
    "../outputs/submissions/submission_moneyness_linear.csv",
    index=False
)

print(submission_moneyness.shape)
submission_moneyness.head()

(5460, 2)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2625500CE,0.11373
1,07-01-2026 09:15||NIFTY27JAN2625800CE,0.10150
2,07-01-2026 09:15||NIFTY27JAN2624100PE,0.16344
3,07-01-2026 09:20||NIFTY27JAN2625300CE,0.09681
4,07-01-2026 09:20||NIFTY27JAN2625400CE,0.10730
